# AksaraScan v4.0 — Multi-Platform GPU Training Notebook

**Kompatibel dengan:** Kaggle Notebooks (`T4 x2` / `P100`), Lightning AI Studios, Saturn Cloud, Paperspace, HuggingFace, & Google Colab.

**Lisensi Bersih (Apache-2.0 / MIT):** Menggunakan arsitektur custom `AksaraYOLOX` (P2 stride-4) + `CRNN-CTC` + `Stage-2 Metric Embedder` — **nol** ketergantungan `ultralytics` (AGPL-3.0) dan **nol** ketergantungan `onnxconverter-common` (konversi FP16 weight storage memakai ONNX standar `< 12 MB`).

**Keamanan Data:** Hanya memproses `aksarascan_v4_gpu_bundle.zip` (`train` + `val` bersih, **tanpa** `test_heldout` atau `eval/heldout_real`).

In [ ]:
import os, glob, shutil, subprocess, zipfile

print('=== 1. Deteksi Platform & GPU ===')
subprocess.run(['nvidia-smi'], check=False)

WORK_DIR = os.path.abspath('./aksarascan_v4_work')
os.makedirs(WORK_DIR, exist_ok=True)

# Cari bundel aksarascan_v4_gpu_bundle.zip otomatis di Kaggle / Colab / Lightning / current dir
candidates = (
    glob.glob('/kaggle/input/**/*aksarascan*bundle*.zip', recursive=True)
    + glob.glob('./*aksarascan*bundle*.zip')
    + glob.glob('../*aksarascan*bundle*.zip')
    + glob.glob('/content/*aksarascan*bundle*.zip')
)

if candidates:
    bundle = candidates[0]
    print('Menemukan bundel:', bundle)
    with zipfile.ZipFile(bundle, 'r') as zf:
        zf.extractall(WORK_DIR)
    os.chdir(WORK_DIR)
else:
    unpacked = glob.glob('/kaggle/input/**/training/train_yolox.py', recursive=True) + glob.glob('./training/train_yolox.py')
    if unpacked:
        src_root = os.path.dirname(os.path.dirname(os.path.abspath(unpacked[0])))
        print('Menemukan bundel terekstrak otomatis di:', src_root)
        if src_root != WORK_DIR:
            for name in os.listdir(src_root):
                sp = os.path.join(src_root, name)
                dp = os.path.join(WORK_DIR, name)
                if os.path.exists(dp) or os.path.islink(dp):
                    continue
                if name == 'dataset' and os.path.isdir(sp):
                    try:
                        os.symlink(sp, dp)
                    except OSError:
                        shutil.copytree(sp, dp)
                elif os.path.isdir(sp):
                    shutil.copytree(sp, dp)
                else:
                    shutil.copy2(sp, dp)
        os.chdir(WORK_DIR)
    else:
        print('Bundel belum ditemukan. Upload aksarascan_v4_gpu_bundle.zip (hasil python tools/build_gpu_bundle.py) ke Input Kaggle/Workspace.')

print('Working directory:', os.getcwd())

In [ ]:
# 2. Verifikasi Dependensi Ringan (tanpa ultralytics / tanpa onnxconverter-common)
!pip install -q onnx onnxruntime PyYAML scipy
import torch, onnx, onnxruntime as ort
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available(), '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# 3. Stage 1 — Pretrain YOLOX-Tiny P2 (640px, AMP GPU, mencakup Hard Negatives Latin + Dinding)
!python -u training/train_yolox.py \
  --variant yolox-tiny \
  --imgsz 640 \
  --stage 1 \
  --epochs 80 \
  --close-mosaic 15 \
  --batch 16 \
  --lr0 0.001 \
  --mosaic 0.25 \
  --seed 42 \
  --workers 2 \
  --amp \
  --export-fp16 \
  --export-int8 \
  --out-dir runs_yolox/v4_stage1_tiny640

In [ ]:
# 4. Stage 2 — Fine-Tune Nyata (Tulisan Tangan + Syllables + Teacher + Hard Negatives)
!python -u training/train_yolox.py \
  --variant yolox-tiny \
  --imgsz 640 \
  --stage 2 \
  --epochs 40 \
  --close-mosaic 10 \
  --batch 12 \
  --lr0 0.00025 \
  --mosaic 0.10 \
  --seed 42 \
  --workers 2 \
  --amp \
  --export-fp16 \
  --export-int8 \
  --resume runs_yolox/v4_stage1_tiny640/yolox-tiny_img640_s42_best.pt \
  --out-dir runs_yolox/v4_stage2_tiny640

In [ ]:
# 5. (Opsional Resolusi Tinggi) Fine-Tune YOLOX-Tiny P2 960px untuk naskah padat / sandhangan tipis
!python -u training/train_yolox.py \
  --variant yolox-tiny \
  --imgsz 960 \
  --stage 2 \
  --epochs 30 \
  --close-mosaic 8 \
  --batch 8 \
  --lr0 0.0002 \
  --mosaic 0.10 \
  --seed 42 \
  --workers 2 \
  --amp \
  --export-fp16 \
  --export-int8 \
  --resume runs_yolox/v4_stage2_tiny640/yolox-tiny_img640_s42_best.pt \
  --out-dir runs_yolox/v4_stage2_tiny960

In [ ]:
# 6. Paketkan Hasil ONNX & Kurva Training untuk Diunduh ke PC Lokal
import zipfile, glob, os

out_root = '/kaggle/working' if os.path.exists('/kaggle/working') else os.getcwd()
out_zip = os.path.join(out_root, 'aksarascan_v4_trained_onnx.zip')
patterns = [
    'runs_yolox/v4_stage*/*_fp16.onnx',
    'runs_yolox/v4_stage*/*_int8.onnx',
    'runs_yolox/v4_stage*/*_history.json',
    'runs_yolox/v4_stage*/*_best.pt',
]
added = 0
with zipfile.ZipFile(out_zip, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for pat in patterns:
        for fp in sorted(glob.glob(pat)):
            zf.write(fp, fp)
            print(f'Added: {fp} ({os.path.getsize(fp)/(1024*1024):.2f} MB)')
            added += 1
print(f'Selesai! Total {added} artefak dikemas di: {out_zip}')
print('Pasang drop-in di PC lokal: python tools/install_yolox_release.py --onnx runs_yolox/v4_stage2_tiny640/yolox-tiny_img640_s42_fp16.onnx')